# Gemma 4 Sparse Autoencoder — Analysis Laboratory

A read-only, publication-oriented view of a completed Gemma 4 E4B sparse-autoencoder run or verified Hugging Face inference release. Hub mode downloads only the SAE release—not Gemma—and never collects activations, trains, evaluates the live language model, or uploads artifacts. Those expensive and state-changing operations belong to the `gemma4-sae` CLI.

The figures below cover provenance, optimization dynamics, held-out reconstruction, sparsity, feature usage, decoder geometry, activating contexts, causal language-model fidelity, and matched base-versus-IT comparisons.

## Analysis contract

- Every repository, revision, cache location, local path, device, and analysis limit is configured in the first code cell. This notebook does not read configuration from environment variables.
- `SOURCE_MODE` explicitly selects the verified Hugging Face release or a local training run; no source takes precedence implicitly.
- Reads cached run artifacts and validation activation shards only when they exist.
- Runs SAE inference on CUDA, Apple MPS, or CPU; `DEVICE = "auto"` selects them in that order.
- Refuses checkpoint/config/activation-manifest mismatches.
- Clearly marks missing artifacts rather than silently recomputing them.
- Excludes text contexts from release packaging unless they have passed a separate privacy and license review.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from IPython.display import HTML, Markdown, display

from gemma4_sae.checkpoint import resolve_checkpoint, validate_checkpoint_provenance
from gemma4_sae.config import load_config
from gemma4_sae.devices import select_device
from gemma4_sae.evaluate import build_sae_from_checkpoint
from gemma4_sae.provenance import canonical_sha256
from gemma4_sae.release import load_release_bundle, resolve_release_bundle
from gemma4_sae.storage import iter_activation_batches

# -----------------------------------------------------------------------------
# ALL NOTEBOOK INPUTS LIVE HERE. No environment variables are read.
# Paths are interpreted relative to REPOSITORY_ROOT unless documented otherwise.
# -----------------------------------------------------------------------------
REPOSITORY_ROOT = "."  # Launch Jupyter from the cloned gemma-sae repository root.
SOURCE_MODE = "huggingface"  # Exactly "huggingface" or "local".

# Hugging Face release source (used only when SOURCE_MODE == "huggingface").
HF_REPO_ID = "lamm-mit/gemma-4-e4b-layer20-batchtopk-sae"
HF_REPO_REVISION = None  # Prefer a commit SHA for a frozen paper artifact.
HF_CACHE_DIR = None  # None uses the standard Hugging Face cache; or set a path here.
HF_LOCAL_FILES_ONLY = False  # True forbids network access and requires a cached snapshot.

# Local training source (used only when SOURCE_MODE == "local").
LOCAL_CONFIG_PATH = "configs/e4b_layer20_batchtopk_dgx_spark_12x_l064.yaml"

# Prompt report shown in section 10. No automatic fallback is performed.
# "release" resolves inside the selected release/run directory; "repository"
# resolves relative to REPOSITORY_ROOT; "disabled" skips the section.
EXPLANATION_BASE = "release"  # Exactly "release", "repository", or "disabled".
EXPLANATION_JSON = "example_explanation.json"
# To inspect a repository-local smoke test instead, set:
# EXPLANATION_BASE = "repository"
# EXPLANATION_JSON = "runs/hub-smoke-test-12x.json"

# Runtime and optional local activation-cache analysis.
DEVICE = "auto"  # auto, cuda, mps, or cpu; explicit unavailable devices fail loudly.
ANALYSIS_BATCH_SIZE = 512  # Lower this if local activation analysis exhausts memory.
MAX_ANALYSIS_BATCHES = 24

ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError(
        f"REPOSITORY_ROOT does not identify the gemma-sae repository: {ROOT}. "
        "Edit REPOSITORY_ROOT in this cell."
    )
analysis_device = select_device(DEVICE)

if SOURCE_MODE not in {"huggingface", "local"}:
    raise ValueError('SOURCE_MODE must be exactly "huggingface" or "local".')
release_mode = SOURCE_MODE == "huggingface"
if release_mode:
    if not HF_REPO_ID.strip():
        raise ValueError("HF_REPO_ID must be set when SOURCE_MODE is 'huggingface'.")
    run_dir = resolve_release_bundle(
        HF_REPO_ID.strip(),
        revision=HF_REPO_REVISION,
        cache_dir=HF_CACHE_DIR,
        local_files_only=HF_LOCAL_FILES_ONLY,
    )
    CONFIG_PATH = run_dir / "resolved_config.json"
    config = load_config(CONFIG_PATH)
    activation_dir = None
    manifest_path = run_dir / "activation_manifest.json"
    feature_report_path = run_dir / "feature_reports.json"
    label_registry_path = run_dir / "feature_labels.json"
    source_label = f"Hugging Face: {HF_REPO_ID}@{HF_REPO_REVISION or 'main'}"
else:
    CONFIG_PATH = ROOT / LOCAL_CONFIG_PATH
    config = load_config(CONFIG_PATH)
    run_dir = ROOT / config.sae.run_dir
    activation_dir = ROOT / config.data.activation_dir
    manifest_path = activation_dir / "manifest.json"
    feature_report_path = run_dir / "feature_reports" / "features.json"
    label_registry_path = run_dir / "feature_labels" / "labels.json"
    source_label = f"Local run: {run_dir}"

sns.set_theme(style="whitegrid", context="talk")
COLORS = {
    "ink": "#172554",
    "blue": "#2563eb",
    "cyan": "#0891b2",
    "gold": "#d97706",
    "red": "#dc2626",
    "green": "#059669",
    "muted": "#64748b",
}
plt.rcParams.update(
    {
        "figure.dpi": 115,
        "savefig.dpi": 300,
        "axes.titleweight": "bold",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

print(f"Notebook repo : {ROOT}")
print(f"Source        : {source_label}")
print(f"Configuration : {CONFIG_PATH}")
print(f"Artifacts     : {run_dir}")
print(f"Model        : {config.model.model_id}@{config.model.revision}")
print(f"Device       : {analysis_device}")

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text()) if Path(path).exists() else None


def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.DataFrame(json.loads(line) for line in path.read_text().splitlines() if line.strip())


def missing_artifact(label, command):
    display(
        HTML(
            f"<div style='padding:14px 18px;border-left:5px solid #d97706;"
            f"background:#fff7ed'><b>{label} is not available.</b><br>"
            f"<code>{command}</code></div>"
        )
    )


def metric_card(label, value, note=""):
    return (
        "<div style='min-width:170px;padding:16px;border:1px solid #dbeafe;"
        "border-radius:12px;background:linear-gradient(135deg,#eff6ff,#fff)'>"
        "<div style='font-size:12px;color:#64748b;text-transform:uppercase;"
        f"letter-spacing:.08em'>{label}</div><div style='font-size:26px;"
        f"font-weight:750;color:#172554'>{value}</div><div style='font-size:12px;"
        f"color:#64748b'>{note}</div></div>"
    )


def artifact_path(relative):
    return run_dir / relative


if release_mode:
    commands = {
        "activations": "Activation shards are intentionally not included in the Hub release.",
        "checkpoint": "Optimizer checkpoints are intentionally not included in the Hub release.",
        "evaluation": "Publish evaluation.json with the release.",
        "fidelity": "Publish fidelity.json with the release.",
        "features": "Publish privacy-reviewed feature reports, or use feature_labels.json.",
    }
else:
    config_label = str(CONFIG_PATH.relative_to(ROOT))
    commands = {
        "activations": f"gemma4-sae collect --config {config_label}",
        "checkpoint": f"gemma4-sae train --config {config_label}",
        "evaluation": f"gemma4-sae evaluate --config {config_label}",
        "fidelity": f"gemma4-sae fidelity --config {config_label}",
        "features": f"gemma4-sae mine --config {config_label}",
    }

## 1. Artifact readiness

The analysis is intentionally incremental: completed CLI stages appear here without triggering missing work.

In [ ]:
artifact_rows = [
    ("Activation manifest", manifest_path, "collect + verify"),
    ("Training history", artifact_path("train_metrics.jsonl"), "train"),
    (
        "SAE weights",
        artifact_path("sae_weights.safetensors")
        if release_mode
        else artifact_path("checkpoints/latest.json"),
        "publish or train",
    ),
    ("Held-out SAE metrics", artifact_path("evaluation.json"), "evaluate"),
    ("Live-model fidelity", artifact_path("fidelity.json"), "fidelity"),
    ("Feature labels", label_registry_path, "develop-labels"),
    ("Feature contexts", feature_report_path, "mine"),
    (
        "Release checksums",
        artifact_path("checksums.json") if release_mode else artifact_path("release"),
        "publish --dry-run",
    ),
]
readiness = pd.DataFrame(
    [
        {
            "artifact": name,
            "ready": path.exists(),
            "producer": producer,
            "path": str(path),
        }
        for name, path, producer in artifact_rows
    ]
)
readiness.style.map(
    lambda value: (
        "color:#059669;font-weight:bold" if value is True else "color:#dc2626;font-weight:bold"
    ),
    subset=["ready"],
).hide(axis="index")

## 2. Experiment identity and provenance

These values should appear verbatim in a paper appendix or model card. A comparison is not controlled if model revision, hook, activation precision, corpus formatting, normalization, or token budget differs unintentionally.

In [ ]:
manifest = read_json(manifest_path)
run_metadata = read_json(artifact_path("run_metadata.json")) or {}
validation_file = read_json(artifact_path("validation_metrics.json")) or {}
evaluation_file = read_json(artifact_path("evaluation.json")) or {}
evaluation_metrics = evaluation_file.get("metrics", evaluation_file)

identity = {
    "model": config.model.model_id,
    "model revision": config.model.revision,
    "hook": f"post-layer residual · decoder layer {config.model.layer_index}",
    "activation dataset": f"{config.data.dataset_id} / {config.data.dataset_config}",
    "dataset revision": config.data.revision,
    "input format": config.data.input_format,
    "dictionary width": (manifest or {}).get("d_model", "—") if manifest else "—",
    "target L0": config.sae.target_l0,
    "seed": config.sae.seed,
    "config SHA-256": canonical_sha256(config.to_dict()),
    "manifest SHA-256": canonical_sha256(manifest) if manifest else "—",
    "repository commit": run_metadata.get("repository_commit", "—"),
}
if manifest:
    identity["residual dimension"] = manifest.get("d_model", "—")
    identity["dictionary width"] = manifest.get("d_model", 0) * config.sae.expansion_factor
    identity["activation tokens"] = manifest.get("total_tokens", "—")
pd.DataFrame({"field": identity.keys(), "value": identity.values()}).style.hide(axis="index")

In [ ]:
headline = {
    "FVE": evaluation_metrics.get("fraction_variance_explained"),
    "normalized MSE": evaluation_metrics.get("normalized_mse"),
    "mean L0": evaluation_metrics.get("mean_l0"),
    "live features": evaluation_metrics.get("active_feature_fraction"),
}
fidelity_file = read_json(artifact_path("fidelity.json")) or {}
fidelity_metrics = fidelity_file.get("metrics", {})
headline["loss recovered"] = fidelity_metrics.get("loss_recovered")


def formatted(value, percent=False):
    if value is None:
        return "—"
    return f"{value:.1%}" if percent else f"{value:.4f}"


cards = [
    metric_card("Held-out FVE", formatted(headline["FVE"], True), "higher is better"),
    metric_card("Normalized MSE", formatted(headline["normalized MSE"]), "lower is better"),
    metric_card("Mean L0", formatted(headline["mean L0"]), f"target {config.sae.target_l0}"),
    metric_card("Live dictionary", formatted(headline["live features"], True), "held-out sample"),
    metric_card(
        "LM loss recovered", formatted(headline["loss recovered"], True), "vs mean ablation"
    ),
]
display(HTML("<div style='display:flex;gap:12px;flex-wrap:wrap'>" + "".join(cards) + "</div>"))

## 3. Optimization dynamics

A healthy run should improve reconstruction without a collapsing dictionary, large L0 drift, or persistent threshold instability. The auxiliary dead-latent loss should help unused features become competitive; optional resampling events are shown over the dead-feature trace.

In [ ]:
history = read_jsonl(artifact_path("train_metrics.jsonl"))
if history.empty:
    missing_artifact("Training history", commands["checkpoint"])
else:
    reconstruction_column = (
        "reconstruction_mse" if "reconstruction_mse" in history else "loss"
    )
    fig, axes = plt.subplots(2, 4, figsize=(22, 9), constrained_layout=True)
    panels = [
        (reconstruction_column, "Reconstruction MSE", COLORS["blue"], None),
        ("auxiliary_loss", "Auxiliary dead-latent loss", COLORS["gold"], None),
        ("fraction_variance_explained", "Fraction variance explained", COLORS["green"], None),
        ("mean_l0", "Mean L0", COLORS["gold"], config.sae.target_l0),
        ("dead_fraction", "Dead feature fraction", COLORS["red"], None),
        ("inference_threshold", "Inference threshold", COLORS["cyan"], None),
        ("learning_rate", "Learning rate", COLORS["muted"], None),
        ("loss", "Total optimization loss", COLORS["ink"], None),
    ]
    for ax, (column, title, color, reference) in zip(axes.flat, panels, strict=True):
        if column not in history:
            ax.text(0.5, 0.5, "not recorded by this run", ha="center", va="center")
            ax.set(title=title, xticks=[], yticks=[])
            continue
        ax.plot(history["tokens_seen"], history[column], color=color, linewidth=2.2)
        if reference is not None:
            ax.axhline(
                reference,
                color=COLORS["ink"],
                linestyle="--",
                linewidth=1.4,
                label=f"target {reference}",
            )
            ax.legend(frameon=False)
        if column == "dead_fraction" and "resampled_features" in history:
            events = history[history["resampled_features"] > 0]
            ax.scatter(
                events["tokens_seen"],
                events[column],
                s=35,
                color=COLORS["gold"],
                label="resampling event",
                zorder=4,
            )
            if len(events):
                ax.legend(frameon=False)
        ax.set(title=title, xlabel="optimizer activation examples")
    fig.suptitle(
        f"Training trajectory · {config.model.model_id} · layer {config.model.layer_index}",
        fontsize=20,
        fontweight="bold",
    )
    plt.show()

## 4. Held-out reconstruction and sparsity

For a local run, the next cell performs bounded, read-only SAE inference over validation shards. It does **not** load Gemma. In Hub mode, it verifies and loads the released SAE weights for geometry analysis; per-token distributions remain unavailable because activation shards are intentionally excluded.

In [ ]:
analysis = None
release_sae = None
decoder_matrix = None
if release_mode:
    release_sae, released_mean, released_scale, released_sae_config = load_release_bundle(
        run_dir, device=analysis_device, verify=False
    )
    decoder_matrix = release_sae.decoder.weight.detach().T.cpu().numpy()
    print(
        f"Loaded {release_sae.n_features:,} verified SAE features from the Hub release. "
        "Per-token distributions require unpublished activation shards."
    )
elif manifest is None or not artifact_path("checkpoints/latest.json").exists():
    missing_artifact("Checkpoint or activation manifest", commands["checkpoint"])
else:
    checkpoint_path = resolve_checkpoint(run_dir, "latest")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    validate_checkpoint_provenance(checkpoint, config.to_dict(), manifest)
    sae = build_sae_from_checkpoint(checkpoint, analysis_device)
    mean = checkpoint["activation_mean"].float()
    scale = checkpoint["activation_scale"].float().clamp_min(1e-8)
    feature_counts = torch.zeros(sae.n_features, dtype=torch.long, device=analysis_device)
    feature_strengths = torch.zeros(sae.n_features, device=analysis_device)
    mse_values, cosine_values, l0_values = [], [], []
    examples = 0
    iterator = iter_activation_batches(
        activation_dir,
        batch_size=ANALYSIS_BATCH_SIZE,
        seed=config.sae.seed + 401,
        validation=True,
        validation_fraction=config.sae.validation_fraction,
        repeat=False,
    )
    with torch.inference_mode():
        for batch_index, batch in enumerate(iterator):
            if batch_index >= MAX_ANALYSIS_BATCHES:
                break
            x = ((batch.activations.float() - mean) / scale).to(analysis_device)
            output = sae(x, use_threshold=True)
            active = output.features > 0
            mse_values.append((output.reconstruction - x).square().mean(dim=-1).cpu())
            cosine_values.append(F.cosine_similarity(output.reconstruction, x, dim=-1).cpu())
            l0_values.append(active.sum(dim=-1).cpu())
            feature_counts += active.sum(dim=0)
            feature_strengths += output.features.sum(dim=0)
            examples += len(x)
    if examples == 0:
        raise RuntimeError(
            "No held-out activation rows. Collect enough shards for the configured "
            "validation fraction."
        )
    analysis = {
        "mse": torch.cat(mse_values).numpy(),
        "cosine": torch.cat(cosine_values).numpy(),
        "l0": torch.cat(l0_values).numpy(),
        "frequency": (feature_counts.float() / examples).cpu().numpy(),
        "mean_strength": (
            feature_strengths / feature_counts.clamp_min(1)
        ).cpu().numpy(),
        "examples": examples,
        "decoder": sae.decoder.weight.detach().T.cpu().numpy(),
    }
    decoder_matrix = analysis["decoder"]
    print(
        f"Analyzed {examples:,} held-out activation vectors from "
        f"{min(MAX_ANALYSIS_BATCHES, batch_index + 1)} batches on {analysis_device}."
    )

In [ ]:
if analysis is not None:
    frame = pd.DataFrame(
        {
            "normalized MSE": analysis["mse"],
            "cosine similarity": analysis["cosine"],
            "L0": analysis["l0"],
        }
    )
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    for ax, column, color in zip(
        axes, frame.columns, [COLORS["blue"], COLORS["green"], COLORS["gold"]], strict=True
    ):
        sns.histplot(
            frame[column],
            bins=50,
            stat="density",
            element="step",
            fill=True,
            alpha=0.35,
            color=color,
            ax=ax,
        )
        ax.axvline(
            frame[column].median(),
            color=COLORS["ink"],
            linestyle="--",
            label=f"median {frame[column].median():.3g}",
        )
        if column == "L0":
            ax.axvline(
                config.sae.target_l0,
                color=COLORS["red"],
                linestyle=":",
                label=f"training target {config.sae.target_l0}",
            )
        ax.legend(frameon=False, fontsize=10)
        ax.set_title(column)
    fig.suptitle("Per-token held-out reconstruction and sparsity", fontsize=19, fontweight="bold")
    plt.show()

## 5. Dictionary utilization and geometry

Frequency concentration reveals dead, rare, and ubiquitous latents. Decoder cosine similarity is estimated from a deterministic feature sample; a heavy positive tail can indicate redundant directions.

In [ ]:
if decoder_matrix is None:
    missing_artifact("SAE decoder geometry", commands["checkpoint"])
else:
    sample_rng = np.random.default_rng(config.sae.seed)
    sample_ids = sample_rng.choice(
        len(decoder_matrix), size=min(768, len(decoder_matrix)), replace=False
    )
    directions = decoder_matrix[sample_ids].copy()
    directions /= np.linalg.norm(directions, axis=1, keepdims=True).clip(1e-12)
    geometry = directions @ directions.T
    pairwise_cosine = geometry[np.triu_indices_from(geometry, k=1)]

    if analysis is not None:
        frequency = analysis["frequency"]
        order = np.argsort(frequency)[::-1]
        positive = frequency[frequency > 0]
        categories = pd.Series(
            {
                "dead": np.mean(frequency == 0),
                "rare (<0.01%)": np.mean((frequency > 0) & (frequency < 1e-4)),
                "mid": np.mean((frequency >= 1e-4) & (frequency <= 0.1)),
                "ubiquitous (>10%)": np.mean(frequency > 0.1),
            }
        )
        fig, axes = plt.subplots(1, 3, figsize=(19, 5.3), constrained_layout=True)
        axes[0].loglog(
            np.arange(1, len(order) + 1),
            np.maximum(frequency[order], 1 / analysis["examples"]),
            color=COLORS["blue"],
            linewidth=2,
        )
        axes[0].set(title="Feature frequency rank", xlabel="feature rank", ylabel="frequency")
        axes[1].bar(
            categories.index,
            categories.values,
            color=[COLORS["red"], COLORS["gold"], COLORS["green"], COLORS["cyan"]],
        )
        axes[1].set(title="Dictionary utilization", ylabel="fraction of features")
        axes[1].tick_params(axis="x", rotation=22)
        geometry_axis = axes[2]
    else:
        fig, geometry_axis = plt.subplots(1, 1, figsize=(10, 5.3), constrained_layout=True)

    sns.histplot(
        pairwise_cosine,
        bins=70,
        stat="density",
        color=COLORS["cyan"],
        alpha=0.38,
        ax=geometry_axis,
    )
    geometry_axis.axvline(0, color=COLORS["ink"], linewidth=1)
    geometry_axis.set(
        title=f"Decoder geometry · {len(sample_ids)} sampled features",
        xlabel="pairwise decoder cosine",
    )
    fig.suptitle("Dictionary utilization and decoder geometry", fontsize=19, fontweight="bold")
    plt.show()

    statistics = {
        "features": len(decoder_matrix),
        "max |sampled pair cosine|": np.max(np.abs(pairwise_cosine)),
        "99th percentile |pair cosine|": np.quantile(np.abs(pairwise_cosine), 0.99),
    }
    if analysis is not None:
        statistics.update(
            {
                "observed-live features": int((frequency > 0).sum()),
                "median positive frequency": np.median(positive) if len(positive) else 0,
            }
        )
    statistics_frame = pd.DataFrame(statistics.items(), columns=["statistic", "value"])
    display(statistics_frame.style.format({"value": "{:.5g}"}).hide(axis="index"))

## 6. Feature hypotheses from held-out contexts

Top contexts are evidence for a hypothesis, not a semantic label. Check diverse positives, random negatives, position/template confounds, and causal effects before naming a feature.

In [ ]:
feature_report = read_json(feature_report_path)
label_registry = read_json(label_registry_path)
if feature_report is None:
    if label_registry is None:
        missing_artifact("Feature labels or held-out contexts", commands["features"])
    else:
        labels = pd.DataFrame(
            {
                "feature": record["feature_id"],
                "status": record["status"],
                "label": record["interpretation"]["label"],
                "confidence": record["interpretation"]["confidence"],
                "description": record["interpretation"]["description"],
            }
            for record in label_registry.get("labels", [])
        )
        display(labels.style.hide(axis="index"))
        print(
            "Released labels are checkpoint-bound hypotheses; raw evidence is "
            "intentionally excluded."
        )
else:
    features = feature_report.get("features", [])
    summary = pd.DataFrame(
        [
            {
                "feature": item["feature_id"],
                "frequency": item["activation_frequency"],
                "top activation": max(
                    (context["activation"] for context in item["top_contexts"]), default=np.nan
                ),
                "contexts": len(item["top_contexts"]),
            }
            for item in features
        ]
    ).sort_values("top activation", ascending=False)
    display(
        summary.style.format({"frequency": "{:.3%}", "top activation": "{:.4f}"})
        .background_gradient(subset=["top activation"], cmap="Blues")
        .hide(axis="index")
    )

    for item in features[: min(6, len(features))]:
        display(
            Markdown(
                f"### Feature {item['feature_id']} · frequency {item['activation_frequency']:.3%}"
            )
        )
        contexts = pd.DataFrame(item["top_contexts"][:8])[["activation", "text"]]
        contexts["text"] = contexts["text"].str.replace("\n", " ↵ ", regex=False).str.slice(0, 220)
        display(
            contexts.style.format({"activation": "{:.4f}"})
            .bar(subset=["activation"], color="#93c5fd")
            .hide(axis="index")
        )

## 7. Causal language-model fidelity

Loss recovered compares SAE reconstruction with a destructive mean-ablation baseline. Paired per-sequence values matter: an acceptable mean can hide a long tail of serious degradation.

In [ ]:
if not fidelity_file:
    missing_artifact("Live-model fidelity report", commands["fidelity"])
else:
    pairs = pd.DataFrame(fidelity_file["per_sequence"])
    metrics = fidelity_file["metrics"]
    pairs["SAE increase"] = pairs["sae_cross_entropy"] - pairs["baseline_cross_entropy"]
    figure, axes = plt.subplots(1, 3, figsize=(19, 5.4), constrained_layout=True)

    subset = pairs.sort_values("baseline_cross_entropy").reset_index(drop=True)
    axes[0].plot(
        subset.index,
        subset["baseline_cross_entropy"],
        label="baseline",
        color=COLORS["ink"],
        linewidth=2,
    )
    axes[0].plot(
        subset.index, subset["sae_cross_entropy"], label="SAE", color=COLORS["blue"], linewidth=2
    )
    axes[0].plot(
        subset.index,
        subset["mean_ablation_cross_entropy"],
        label="mean ablation",
        color=COLORS["red"],
        linewidth=2,
        alpha=0.8,
    )
    axes[0].set(
        title="Paired sequence cross-entropy",
        xlabel="sequence, ordered by baseline",
        ylabel="cross-entropy",
    )
    axes[0].legend(frameon=False, fontsize=10)

    sns.histplot(
        pairs["SAE increase"],
        bins=min(30, max(8, len(pairs) // 2)),
        color=COLORS["blue"],
        alpha=0.4,
        ax=axes[1],
    )
    axes[1].axvline(0, color=COLORS["ink"], linewidth=1)
    axes[1].axvline(
        pairs["SAE increase"].mean(),
        color=COLORS["red"],
        linestyle="--",
        label=f"mean {pairs['SAE increase'].mean():.4f}",
    )
    axes[1].set(title="Per-sequence SAE damage", xlabel="SAE cross-entropy − baseline")
    axes[1].legend(frameon=False, fontsize=10)

    names = ["baseline", "SAE", "mean ablation"]
    values = [
        metrics["baseline_cross_entropy"],
        metrics["sae_cross_entropy"],
        metrics["mean_ablation_cross_entropy"],
    ]
    axes[2].bar(names, values, color=[COLORS["ink"], COLORS["blue"], COLORS["red"]])
    axes[2].set(
        title=f"Loss recovered: {metrics['loss_recovered']:.1%}", ylabel="mean cross-entropy"
    )
    axes[2].tick_params(axis="x", rotation=18)
    figure.suptitle(
        "Does replacing the residual stream with the SAE preserve Gemma?",
        fontsize=19,
        fontweight="bold",
    )
    plt.show()

    ci = pd.DataFrame(
        {
            "metric": ["SAE CE increase", "loss recovered"],
            "estimate": [metrics["sae_cross_entropy_increase"], metrics["loss_recovered"]],
            "95% CI lower": [
                metrics["sae_cross_entropy_increase_95ci"][0],
                metrics["loss_recovered_95ci"][0],
            ],
            "95% CI upper": [
                metrics["sae_cross_entropy_increase_95ci"][1],
                metrics["loss_recovered_95ci"][1],
            ],
        }
    )
    display(
        ci.style.format({column: "{:.5f}" for column in ci.columns if column != "metric"}).hide(
            axis="index"
        )
    )

## 8. Matched base versus instruction-tuned comparison

This panel reads both checked-in run directories when available. Interpret it only when token budgets, layer semantics, width/L0 grids, seeds, and evaluation protocols are deliberately matched.

In [ ]:
comparison_rows = []
if release_mode:
    comparison_rows.append(
        {
            "variant": "selected Hub release",
            "model": config.model.model_id,
            "FVE": evaluation_metrics.get("fraction_variance_explained"),
            "mean L0": evaluation_metrics.get("mean_l0"),
            "live feature fraction": evaluation_metrics.get("active_feature_fraction"),
            "SAE CE increase": fidelity_metrics.get("sae_cross_entropy_increase"),
            "loss recovered": fidelity_metrics.get("loss_recovered"),
        }
    )
comparison_candidates = [] if release_mode else [
    ("base", "configs/e4b_layer20_batchtopk.yaml"),
    ("instruction tuned", "configs/e4b_it_layer20_batchtopk.yaml"),
]
for label, relative_config in comparison_candidates:
    candidate = load_config(ROOT / relative_config)
    candidate_run = ROOT / candidate.sae.run_dir
    candidate_eval = read_json(candidate_run / "evaluation.json") or {}
    candidate_fidelity = read_json(candidate_run / "fidelity.json") or {}
    em = candidate_eval.get("metrics", {})
    fm = candidate_fidelity.get("metrics", {})
    comparison_rows.append(
        {
            "variant": label,
            "model": candidate.model.model_id,
            "FVE": em.get("fraction_variance_explained"),
            "mean L0": em.get("mean_l0"),
            "live feature fraction": em.get("active_feature_fraction"),
            "SAE CE increase": fm.get("sae_cross_entropy_increase"),
            "loss recovered": fm.get("loss_recovered"),
        }
    )

comparison = pd.DataFrame(comparison_rows)
display(
    comparison.style.format(
        {
            "FVE": "{:.2%}",
            "mean L0": "{:.2f}",
            "live feature fraction": "{:.2%}",
            "SAE CE increase": "{:.5f}",
            "loss recovered": "{:.2%}",
        },
        na_rep="—",
    ).hide(axis="index")
)
if len(comparison) >= 2 and comparison[["FVE", "mean L0", "loss recovered"]].notna().all().all():
    plot_frame = comparison.set_index("variant")[["FVE", "live feature fraction", "loss recovered"]]
    axes = plot_frame.plot(
        kind="bar",
        subplots=True,
        layout=(1, 3),
        figsize=(17, 4.8),
        legend=False,
        color=[COLORS["blue"], COLORS["cyan"]],
        rot=0,
    )
    for ax, title in zip(
        np.asarray(axes).flat,
        ["Held-out FVE", "Live dictionary fraction", "LM loss recovered"],
        strict=True,
    ):
        ax.set_title(title)
        ax.set_xlabel("")
    plt.suptitle("Base versus instruction-tuned SAE diagnostics", fontsize=19, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(
        "Complete evaluate + fidelity for both configurations to render the matched "
        "comparison figure."
    )

## 9. Publication evidence gate

Passing this table means the single run is auditable; it does not establish that the scientific study is complete. Multi-seed sweeps, feature quality, stability, causal tests, failed-run disclosure, and privacy/license review still require study-level evidence.

In [ ]:
checks = [
    ("Model revision pinned", bool(config.model.revision), config.model.revision),
    ("Dataset revision pinned", bool(config.data.revision), config.data.revision),
    ("Activation manifest recorded", manifest is not None, str(manifest_path)),
    ("Shard hashes enabled", config.data.hash_shards, "data.hash_shards"),
    (
        "BatchTopK auxiliary objective enabled",
        config.sae.auxiliary_loss_coefficient > 0 and config.sae.auxiliary_top_k > 0,
        f"top-k={config.sae.auxiliary_top_k}, "
        f"coefficient={config.sae.auxiliary_loss_coefficient:g}",
    ),
    (
        "Repository commit recorded",
        bool(run_metadata.get("repository_commit")),
        run_metadata.get("repository_commit", "missing"),
    ),
    ("Held-out reconstruction complete", bool(evaluation_file), "evaluation.json"),
    (
        "Held-out dictionary coverage passes release gate",
        evaluation_metrics.get("active_feature_fraction", 0)
        >= config.publication.min_active_feature_fraction,
        f"minimum={config.publication.min_active_feature_fraction:.1%}",
    ),
    ("Independent LM fidelity complete", bool(fidelity_file), "fidelity.json"),
    (
        "Bootstrap confidence intervals",
        "loss_recovered_95ci" in fidelity_metrics,
        "fidelity metrics",
    ),
    (
        "Feature contexts available",
        feature_report is not None,
        "privacy review required before release",
    ),
    (
        "Hugging Face namespace configured",
        config.publication.hf_repo_id.startswith("lamm-mit/"),
        config.publication.hf_repo_id,
    ),
    (
        "Text contexts excluded by default",
        not config.publication.include_feature_reports,
        "publication.include_feature_reports",
    ),
]
gate = pd.DataFrame(checks, columns=["evidence", "pass", "detail"])
display(
    gate.style.map(
        lambda value: (
            "background:#dcfce7;color:#166534;font-weight:bold"
            if value is True
            else "background:#fee2e2;color:#991b1b;font-weight:bold"
        ),
        subset=["pass"],
    ).hide(axis="index")
)
print(f"Run-level evidence: {gate['pass'].sum()}/{len(gate)} checks available")

## 10. New-prompt feature explanation

This read-only notebook does not run Gemma or create prompt reports. `EXPLANATION_BASE` and `EXPLANATION_JSON` in the first code cell identify the report explicitly. The default is the checksummed `example_explanation.json` inside the selected Hugging Face release. Set the base to `repository` and provide a repository-relative path to inspect another report, or set it to `disabled` to skip this section. There is no path discovery or fallback. This view summarizes prompt L0, label coverage, token-level sparsity, and compatible checkpoint-bound labels. Labels remain hypotheses whose held-out metrics and contexts require review.

In [ ]:
valid_explanation_bases = {"release", "repository", "disabled"}
if EXPLANATION_BASE not in valid_explanation_bases:
    raise ValueError(
        "EXPLANATION_BASE must be exactly 'release', 'repository', or 'disabled'."
    )

if EXPLANATION_BASE == "disabled":
    explanation_path = None
    display(
        Markdown(
            "*Prompt-report analysis is disabled in the first configuration cell.*"
        )
    )
else:
    if not EXPLANATION_JSON.strip():
        raise ValueError(
            "EXPLANATION_JSON must name a file unless EXPLANATION_BASE is 'disabled'."
        )
    explanation_path = Path(EXPLANATION_JSON).expanduser()
    if not explanation_path.is_absolute():
        base_dir = run_dir if EXPLANATION_BASE == "release" else ROOT
        explanation_path = base_dir / explanation_path
    explanation_path = explanation_path.resolve()
    if not explanation_path.is_file():
        raise FileNotFoundError(
            f"Configured prompt report does not exist: {explanation_path}. "
            "Edit EXPLANATION_BASE and EXPLANATION_JSON in the first code cell."
        )
    print(f"Using configured explanation report: {explanation_path}")
    explanation = read_json(explanation_path)
    if explanation is None:
        raise FileNotFoundError(explanation_path)

    display(Markdown(f"### Prompt\n\n> {explanation['prompt']}"))
    tokens = explanation["tokens"]
    ordinary_tokens = [token for token in tokens if not token["special"]]
    non_special_mean_l0 = (
        float(np.mean([token["active_feature_count"] for token in ordinary_tokens]))
        if ordinary_tokens
        else float("nan")
    )
    prompt_feature_rows = explanation["prompt_features"]
    labeled_features = sum(
        bool(feature.get("interpretation")) for feature in prompt_feature_rows
    )
    display(
        HTML(
            "<div style='display:flex;gap:12px;flex-wrap:wrap;margin:10px 0 18px'>"
            + metric_card(
                "Mean prompt L0",
                f"{explanation['mean_inference_l0']:.1f}",
                "all prompt tokens",
            )
            + metric_card(
                "Non-special L0",
                f"{non_special_mean_l0:.1f}",
                "mean across ordinary tokens",
            )
            + metric_card(
                "Inference threshold",
                f"{explanation['inference_threshold']:.3f}",
                "learned global threshold",
            )
            + metric_card(
                "Labeled top features",
                f"{labeled_features}/{len(prompt_feature_rows)}",
                f"{explanation['labeled_prompt_feature_fraction']:.0%} coverage",
            )
            + "</div>"
        )
    )
    heldout_mean_l0 = evaluation_metrics.get("mean_l0")
    if heldout_mean_l0 is not None:
        display(
            Markdown(
                f"The released held-out mean L0 is **{heldout_mean_l0:.1f}**, while this "
                f"prompt's mean is **{explanation['mean_inference_l0']:.1f}**. BatchTopK "
                "controls an average during training; prompt inference uses one learned "
                "global threshold, so token- and prompt-level L0 can vary. A higher value "
                "on one short or domain-specific prompt is not by itself a release failure."
            )
        )

    token_l0 = pd.DataFrame(
        {
            "position": [token["position"] for token in tokens],
            "token": [token["text"] or token["token"] for token in tokens],
            "active features (L0)": [
                token["active_feature_count"] for token in tokens
            ],
            "special": [token["special"] for token in tokens],
        }
    )
    display(
        token_l0.style.background_gradient(
            subset=["active features (L0)"], cmap="YlOrBr"
        ).hide(axis="index")
    )
    plt.figure(figsize=(max(12, min(28, len(tokens) * 0.65)), 5))
    sns.barplot(
        data=token_l0,
        x="position",
        y="active features (L0)",
        hue="special",
        palette={False: COLORS["blue"], True: COLORS["gold"]},
    )
    if heldout_mean_l0 is not None:
        plt.axhline(
            heldout_mean_l0,
            color=COLORS["red"],
            linestyle="--",
            linewidth=2,
            label=f"held-out mean L0 = {heldout_mean_l0:.1f}",
        )
    plt.title("Threshold-active SAE features per prompt token")
    plt.xlabel("token position")
    plt.ylabel("active features (L0)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    prompt_summary = pd.DataFrame(prompt_feature_rows)
    if not prompt_summary.empty:
        if "interpretation" not in prompt_summary:
            prompt_summary["interpretation"] = None
        prompt_summary["known context examples"] = prompt_summary["known_contexts"].map(len)
        prompt_summary["label"] = prompt_summary["interpretation"].map(
            lambda value: value.get("label", "unlabeled") if value else "unlabeled"
        )
        prompt_summary["label status"] = prompt_summary["interpretation"].map(
            lambda value: value.get("status", "unlabeled") if value else "unlabeled"
        )
        display(
            prompt_summary[
                [
                    "feature_id",
                    "label",
                    "label status",
                    "max_activation",
                    "mean_active_activation",
                    "active_token_count",
                    "token_positions",
                    "known context examples",
                ]
            ]
            .style.format(
                {"max_activation": "{:.4f}", "mean_active_activation": "{:.4f}"}
            )
            .background_gradient(subset=["max_activation"], cmap="Blues")
            .hide(axis="index")
        )

        feature_ids = prompt_summary["feature_id"].head(20).astype(int).tolist()
        feature_columns = {feature_id: index for index, feature_id in enumerate(feature_ids)}
        heatmap = np.zeros((len(tokens), len(feature_ids)), dtype=np.float32)
        for token_index, token in enumerate(tokens):
            for feature in token["top_features"]:
                column = feature_columns.get(feature["feature_id"])
                if column is not None:
                    heatmap[token_index, column] = feature["activation"]
        token_labels = [
            f"{token['position']}:{token['text'] or token['token']}" for token in tokens
        ]
        figure_width = max(12, min(28, len(tokens) * 0.55))
        plt.figure(figsize=(figure_width, max(6, len(feature_ids) * 0.34)))
        sns.heatmap(
            heatmap.T,
            xticklabels=token_labels,
            yticklabels=[f"F{feature_id}" for feature_id in feature_ids],
            cmap="mako",
            linewidths=0.25,
            cbar_kws={"label": "SAE activation"},
        )
        plt.title("Which SAE features activate on each prompt token?")
        plt.xlabel("token position and decoded token")
        plt.ylabel("feature ID")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("The explanation contains no threshold-active prompt features.")

## Producing missing artifacts and publishing

Run expensive stages in a terminal, ideally through a scheduler or persistent session:

```bash
CONFIG=configs/e4b_layer20_batchtopk_dgx_spark_12x_l064.yaml
gemma4-sae doctor   --config "$CONFIG"
gemma4-sae verify   --config "$CONFIG"
gemma4-sae train    --config "$CONFIG"  # add --resume only after a checkpoint exists
gemma4-sae evaluate --config "$CONFIG" --checkpoint latest --max-batches 1000000
gemma4-sae fidelity --config "$CONFIG" --checkpoint latest
gemma4-sae explain  --config "$CONFIG" --checkpoint latest --text "Paris is the capital of France." --output runs/prompt-paris.json
gemma4-sae mine     --config "$CONFIG" --checkpoint latest --n-features 64 --top-contexts 40 --random-contexts 40 --max-batches 4096
gemma4-sae label    --config "$CONFIG" --checkpoint latest --provider transformers --model "ORG/INSTRUCT-MODEL"
```

Package and inspect the exact release locally without network writes:

```bash
gemma4-sae publish --config "$CONFIG" --checkpoint latest --dry-run
```

After the release card, metrics, context privacy, data terms, and organization permissions have been reviewed, upload publicly to the configured `lamm-mit` repository:

```bash
export HF_TOKEN=hf_...  # write permission to lamm-mit; never commit it
gemma4-sae publish --config "$CONFIG" --checkpoint latest --repo-id lamm-mit/gemma-4-e4b-layer20-batchtopk-sae --public
```

The uploader sends inference-only `safetensors`, configuration, provenance, aggregate metrics, a model card, and SHA-256 checksums. Optimizer state and mined text contexts are excluded by default.

On another machine, paste the published repository ID into `HF_REPO_ID` in the first code cell. Explain a new prompt directly from the verified release with:

```bash
gemma4-sae explain \
  --sae-repo lamm-mit/gemma-4-e4b-layer20-batchtopk-sae \
  --device auto \
  --text "Paris is the capital of France." \
  --output prompt-paris.json
```

## Interpretation boundary

An SAE can be useful even when its representation is non-unique. Features may split, merge, duplicate, track formatting artifacts, or change across random seeds. Strong reconstruction is necessary for many uses but does not prove monosemanticity. Treat every feature name as a falsifiable hypothesis supported by held-out examples, negative controls, stability analysis, and causal intervention.